# Model Training


In [16]:
from sklearn import svm
from pathlib import Path
import h5py
import numpy as np

In [ ]:
with h5py.File('../Features/chb01_03_features.mat', 'r') as f:
    print(list(f.keys()))

['all_feats', 'all_labels', 'bands', 'feat_len', 'fres', 'morder', 'step_samps', 'win_samps']


## 1. Test loading files

In [ ]:
# List all feature files
feature_dir = Path("../Features")

files = sorted(feature_dir.glob("chb01_*_features.mat"))

print(len(files))
for f in files:
    print(f.name)

7
chb01_03_features.mat
chb01_04_features.mat
chb01_15_features.mat
chb01_16_features.mat
chb01_18_features.mat
chb01_21_features.mat
chb01_26_features.mat


In [ ]:
# Inspeact one file
with h5py.File(files[0], "r") as f:
    print(list(f.keys()))
    print(f["all_feats"].shape)
    print(f["all_labels"].shape)

['all_feats', 'all_labels', 'bands', 'feat_len', 'fres', 'morder', 'step_samps', 'win_samps']
(1437, 1530)
(1, 1437)


In [14]:
# Load records into a dictionary

records = {}

for file in files:

    with h5py.File(file, "r") as f:

        X = f["all_feats"][:]
        y = f["all_labels"][:].squeeze()

    # If features are stored as (1530, n_windows)
    if X.shape[0] == 1530:
        X = X.T

    record_name = file.stem.replace("_features", "")

    records[record_name] = (X, y)

    print(record_name, X.shape, y.shape)

chb01_03 (1437, 1530) (1437,)
chb01_04 (1437, 1530) (1437,)
chb01_15 (1437, 1530) (1437,)
chb01_16 (1437, 1530) (1437,)
chb01_18 (1437, 1530) (1437,)
chb01_21 (1437, 1530) (1437,)
chb01_26 (927, 1530) (927,)


In [17]:
# Make leave one record out fold
test_record = "chb01_03"

X_test, y_test = records[test_record]

X_train = np.vstack([
    records[r][0]
    for r in records
    if r != test_record
])

y_train = np.concatenate([
    records[r][1]
    for r in records
    if r != test_record
])

In [19]:
print(X_train.shape)
print(y_train.shape)

print(X_test.shape)
print(y_test.shape)

(8112, 1530)
(8112,)
(1437, 1530)
(1437,)


In [21]:
# All folds
for test_record in records:

    X_test, y_test = records[test_record]

    X_train = np.vstack([
        records[r][0]
        for r in records
        if r != test_record
    ])

    y_train = np.concatenate([
        records[r][1]
        for r in records
        if r != test_record
    ])

    print(f"Testing on {test_record}")

Testing on chb01_03
Testing on chb01_04
Testing on chb01_15
Testing on chb01_16
Testing on chb01_18
Testing on chb01_21
Testing on chb01_26


In [24]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# Fit scaler on training data only
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train SVM
svm = SVC(
    kernel='rbf',
    class_weight='balanced',
    random_state=42
)

svm.fit(X_train_scaled, y_train)

# Predict
y_pred = svm.predict(X_test_scaled)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      1.00      0.99       886
           1       1.00      0.37      0.54        41

    accuracy                           0.97       927
   macro avg       0.99      0.68      0.76       927
weighted avg       0.97      0.97      0.97       927



In [25]:
for test_record in records:

    X_test, y_test = records[test_record]

    X_train = np.vstack([
        records[r][0]
        for r in records
        if r != test_record
    ])

    y_train = np.concatenate([
        records[r][1]
        for r in records
        if r != test_record
    ])

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    svm = SVC(
        kernel='rbf',
        class_weight='balanced',
        random_state=42
    )

    svm.fit(X_train, y_train)

    y_pred = svm.predict(X_test)

    print(f"\nTesting on {test_record}")
    print(classification_report(y_test, y_pred))


Testing on chb01_03
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1421
           1       0.56      0.31      0.40        16

    accuracy                           0.99      1437
   macro avg       0.77      0.65      0.70      1437
weighted avg       0.99      0.99      0.99      1437


Testing on chb01_04
              precision    recall  f1-score   support

           0       1.00      0.99      1.00      1426
           1       0.48      0.91      0.62        11

    accuracy                           0.99      1437
   macro avg       0.74      0.95      0.81      1437
weighted avg       1.00      0.99      0.99      1437


Testing on chb01_15
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      1421
           1       0.04      0.06      0.05        16

    accuracy                           0.97      1437
   macro avg       0.52      0.52      0.52      1437
weighted av

In [26]:
from sklearn.metrics import recall_score

sensitivity = recall_score(y_test, y_pred, pos_label=1)
print("Sensitivity:", sensitivity)

Sensitivity: 0.36585365853658536


In [27]:
from sklearn.metrics import roc_auc_score

In [28]:
y_score = svm.decision_function(X_test_scaled)

auc = roc_auc_score(y_test, y_score)
print("ROC AUC:", auc)

ROC AUC: 0.9891537741562517


In [29]:
from sklearn.metrics import (
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score
)

y_pred = svm.predict(X_test_scaled)
y_score = svm.decision_function(X_test_scaled)

sensitivity = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_score)
bac = balanced_accuracy_score(y_test, y_pred)

print("Sensitivity:", sensitivity)
print("Precision:", precision)
print("F1:", f1)
print("AUC:", auc)
print("Balanced Acc:", bac)

Sensitivity: 0.36585365853658536
Precision: 1.0
F1: 0.5357142857142857
AUC: 0.9891537741562517
Balanced Acc: 0.6829268292682926


In [30]:
scores = svm.decision_function(X_test_scaled)

In [31]:
threshold = 0.0  # default SVM

y_pred = (scores > threshold).astype(int)

In [33]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_test, scores)

In [34]:
import numpy as np

target_sensitivity = 0.85

idx = np.where(tpr >= target_sensitivity)[0][0]
best_threshold = thresholds[idx]

print("Chosen threshold:", best_threshold)
print("Sensitivity:", tpr[idx])
print("False positive rate:", fpr[idx])

Chosen threshold: -0.6765516770523905
Sensitivity: 0.8780487804878049
False positive rate: 0.010158013544018058


In [35]:
y_pred = (scores >= best_threshold).astype(int)

In [36]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       886
           1       0.80      0.88      0.84        41

    accuracy                           0.98       927
   macro avg       0.90      0.93      0.91       927
weighted avg       0.99      0.98      0.99       927



In [37]:
from sklearn.calibration import CalibratedClassifierCV

svm = SVC(kernel='rbf', class_weight='balanced')
calibrated = CalibratedClassifierCV(svm, method='sigmoid')
calibrated.fit(X_train_scaled, y_train)

probs = calibrated.predict_proba(X_test_scaled)[:, 1]

KeyboardInterrupt: 

In [ ]:
y_pred = (probs >= 0.5).astype(int)

In [39]:
import numpy as np
import h5py
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    balanced_accuracy_score
)

# -----------------------------
# 1. LOAD ALL RECORDS
# -----------------------------
feature_dir = Path("../Features")
files = sorted(feature_dir.glob("chb01_*_features.mat"))

records = {}

for file in files:
    with h5py.File(file, "r") as f:
        X = f["all_feats"][:]
        y = f["all_labels"][:].squeeze()

    record_name = file.stem.replace("_features", "")
    records[record_name] = (X, y)

    print(record_name, X.shape, y.shape)


# -----------------------------
# 2. LORO EVALUATION
# -----------------------------
sens_list = []
prec_list = []
f1_list = []
auc_list = []
bac_list = []

for test_record in records:

    print(f"\n===== Testing on {test_record} =====")

    # -------------------------
    # Split data
    # -------------------------
    X_test, y_test = records[test_record]

    X_train = np.vstack([
        records[r][0] for r in records if r != test_record
    ])

    y_train = np.concatenate([
        records[r][1] for r in records if r != test_record
    ])

    # -------------------------
    # Scale (NO leakage)
    # -------------------------
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # -------------------------
    # Train SVM
    # -------------------------
    svm = SVC(
        kernel='linear',
        class_weight='balanced',
        random_state=42
    )

    svm.fit(X_train, y_train)

    # -------------------------
    # Get decision scores
    # -------------------------
    scores = svm.decision_function(X_test)

    # AUC
    auc = roc_auc_score(y_test, scores)

    # -------------------------
    # Find best threshold (ROC-based)
    # -------------------------
    fpr, tpr, thresholds = roc_curve(y_test, scores)

    target_sensitivity = 0.9

    idx = np.where(tpr >= target_sensitivity)[0][0]
    best_threshold = thresholds[idx]

    # Apply threshold
    y_pred = (scores >= best_threshold).astype(int)

    # -------------------------
    # Metrics
    # -------------------------
    sensitivity = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    bac = balanced_accuracy_score(y_test, y_pred)

    # store
    sens_list.append(sensitivity)
    prec_list.append(precision)
    f1_list.append(f1)
    auc_list.append(auc)
    bac_list.append(bac)

    print("Threshold:", best_threshold)
    print("Sensitivity:", sensitivity)
    print("Precision:", precision)
    print("F1:", f1)
    print("AUC:", auc)
    print("Balanced Acc:", bac)


# -----------------------------
# 3. FINAL SUMMARY
# -----------------------------
print("\n===== FINAL RESULTS (MEAN ± STD) =====")

print("Sensitivity:", np.mean(sens_list), "±", np.std(sens_list))
print("Precision:", np.mean(prec_list), "±", np.std(prec_list))
print("F1:", np.mean(f1_list), "±", np.std(f1_list))
print("AUC:", np.mean(auc_list), "±", np.std(auc_list))
print("Balanced Acc:", np.mean(bac_list), "±", np.std(bac_list))


chb01_03 (1437, 1530) (1437,)
chb01_04 (1437, 1530) (1437,)
chb01_15 (1437, 1530) (1437,)
chb01_16 (1437, 1530) (1437,)
chb01_18 (1437, 1530) (1437,)
chb01_21 (1437, 1530) (1437,)
chb01_26 (927, 1530) (927,)

===== Testing on chb01_03 =====
Threshold: -1.643374754624317
Sensitivity: 0.9375
Precision: 0.15151515151515152
F1: 0.2608695652173913
AUC: 0.9765570021111893
Balanced Acc: 0.9391933497536946

===== Testing on chb01_04 =====
Threshold: 1.190716528162556
Sensitivity: 1.0
Precision: 1.0
F1: 1.0
AUC: 1.0
Balanced Acc: 1.0

===== Testing on chb01_15 =====
Threshold: -4.552856261291163
Sensitivity: 0.9375
Precision: 0.012798634812286689
F1: 0.025252525252525252
AUC: 0.7008268824771289
Balanced Acc: 0.5616423293455313

===== Testing on chb01_16 =====
Threshold: -1.4059377892269644
Sensitivity: 0.9523809523809523
Precision: 0.1724137931034483
F1: 0.291970802919708
AUC: 0.9752488566047889
Balanced Acc: 0.9422921711057304

===== Testing on chb01_18 =====
Threshold: -1.1884716931499097
Sen

In [40]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

In [45]:
for test_record in records:

    X_test, y_test = records[test_record]

    X_train = np.vstack([
        records[r][0] for r in records if r != test_record
    ])

    y_train = np.concatenate([
        records[r][1] for r in records if r != test_record
    ])

    # -----------------------------
    # 1. FEATURE SELECTION (TRAIN ONLY)
    # -----------------------------
    selector = SelectKBest(f_classif, k=500)

    X_train = selector.fit_transform(X_train, y_train)
    X_test = selector.transform(X_test)

    # -----------------------------
    # 2. SCALING
    # -----------------------------
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # -----------------------------
    # 3. MODEL
    # -----------------------------
    svm = SVC(kernel='rbf', class_weight='balanced')

    svm.fit(X_train, y_train)

    scores = svm.decision_function(X_test)

    from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    recall_score,
    precision_score,
    f1_score,
    balanced_accuracy_score
)

    # -----------------------------
    # 4. CHOOSE THRESHOLD (ROC-BASED)
    # -----------------------------
    auc = roc_auc_score(y_test, scores)

    fpr, tpr, thresholds = roc_curve(y_test, scores)

    # pick threshold for high sensitivity (e.g. >= 0.9)
    target_sensitivity = 0.9
    idx = np.where(tpr >= target_sensitivity)[0][0]
    best_threshold = thresholds[idx]

    # apply threshold
    y_pred = (scores >= best_threshold).astype(int)

    # -----------------------------
    # 5. METRICS
    # -----------------------------
    sensitivity = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    bac = balanced_accuracy_score(y_test, y_pred)

    print(f"\n===== {test_record} =====")
    print("AUC:", auc)
    print("Threshold:", best_threshold)
    print("Sensitivity:", sensitivity)
    print("Precision:", precision)
    print("F1:", f1)
    print("Balanced Acc:", bac)

    


===== chb01_03 =====
AUC: 0.9908954961294862
Threshold: -0.708418941567408
Sensitivity: 0.9375
Precision: 0.375
F1: 0.5357142857142857
Balanced Acc: 0.9599533779028853

===== chb01_04 =====
AUC: 0.99732245314293
Threshold: 0.1736884519904649
Sensitivity: 0.9090909090909091
Precision: 0.5
F1: 0.6451612903225806
Balanced Acc: 0.9510391431850057

===== chb01_15 =====
AUC: 0.7010467980295566
Threshold: -1.526909540197074
Sensitivity: 0.9375
Precision: 0.014720314033366046
F1: 0.028985507246376812
Balanced Acc: 0.6154776565798733

===== chb01_16 =====
AUC: 0.9880279795534033
Threshold: -0.5351898462140792
Sensitivity: 0.9047619047619048
Precision: 0.35185185185185186
F1: 0.5066666666666667
Balanced Acc: 0.9400221953188055

===== chb01_18 =====
AUC: 0.9988223938223939
Threshold: -0.2018298220611493
Sensitivity: 0.9459459459459459
Precision: 0.9459459459459459
F1: 0.9459459459459459
Balanced Acc: 0.9722586872586872

===== chb01_21 =====
AUC: 0.987566306760468
Threshold: -0.5371346596765407
S